# PanganPulse — ETL, Normalisasi Komoditas, dan Persiapan Clustering


## Alur kerja
1. Konfigurasi lokasi data dan library.
2. Memeriksa file sumber dan struktur kolom.
3. Mengubah data wide menjadi long, membersihkan harga, dan menghapus duplikasi.
4. Menggabungkan hasil ETL 2025 dan 2026.
5. Mengevaluasi kualitas data dan daftar komoditas.
6. Menyusun kandidat mapping nama komoditas lintas tahun.
7. Membandingkan harga kandidat pada wilayah dan periode yang sebanding.
8. Mengisi mapping final setelah hasilnya ditinjau.
9. Membentuk matriks awal wilayah × komoditas untuk clustering dan memeriksa missing value.

## Struktur folder input yang diharapkan

```text
PanganPulse/
└── etl/
    └── data/
        └── dataset/
            ├── januari.csv
            ├── februari.csv
            ├── ...
            ├── desember.csv
            └── data-2026.csv
```

Notebook mencari folder `etl/data/dataset` jika dijalankan dari root proyek, atau `data/dataset` jika dijalankan dari folder `etl`.

## Catatan metodologis
- Data sumber tidak diubah. Hasil bersih disimpan sebagai file baru di folder `processed`.
- Normalisasi dasar hanya merapikan spasi dan huruf kapital. Nama yang mirip **tidak otomatis** dianggap komoditas yang sama.
- Harga dipakai sebagai bukti pendukung untuk menilai kandidat mapping, bukan satu-satunya bukti kesamaan produk.
- Matriks clustering di bagian akhir masih berupa matriks awal. Notebook ini belum melakukan imputasi atau menjalankan K-Means.

## 1. Import library dan konfigurasi

In [25]:
from pathlib import Path
import pandas as pd
import numpy as np

from IPython.display import display

# SciPy digunakan untuk uji Wilcoxon dan korelasi Spearman.
# Jika belum terpasang, notebook tetap berjalan; kolom p-value akan kosong.
try:
    from scipy.stats import wilcoxon, spearmanr
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    print("SciPy belum tersedia. Instal dengan: pip install scipy")

MONTHS = [
    'januari', 'februari', 'maret', 'april', 'mei', 'juni',
    'juli', 'agustus', 'september', 'oktober', 'november', 'desember'
]

REQUIRED_COLUMNS = [
    'kode_daerah', 'provinsi', 'kabupaten_kota', 'komoditas'
]

IDENTIFIER_COLUMNS = [
    'kode_provinsi', 'provinsi', 'kode_daerah',
    'kabupaten_kota', 'kode_komoditas', 'komoditas'
]

COLUMNS_RENAME = {
    'Kode Provinsi': 'kode_provinsi',
    'Provinsi': 'provinsi',
    'Kode Daerah': 'kode_daerah',
    'Kode Wilayah': 'kode_daerah',
    'Kabupaten Kota': 'kabupaten_kota',
    'COICOP': 'kode_komoditas',
    'Komoditas': 'komoditas',
}

# Cari folder dataset berdasarkan lokasi notebook dijalankan.
DATA_DIR_CANDIDATES = [
    Path.cwd() / 'etl' / 'data' / 'dataset',
    Path.cwd() / 'data' / 'dataset',
]

DATA_DIR = next(
    (path.resolve() for path in DATA_DIR_CANDIDATES if path.is_dir()),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Folder dataset tidak ditemukan. Pastikan file CSV berada di "
        "'etl/data/dataset' (dari root proyek) atau 'data/dataset' (dari folder etl)."
    )

OUTPUT_DIR = DATA_DIR.parent / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILES = [DATA_DIR / f'{month}.csv' for month in MONTHS]
INPUT_FILES.append(DATA_DIR / 'data-2026.csv')

# Format sumber 2025 dan 2026 mengikuti file yang dipakai pada ETL sebelumnya.
SOURCE_FORMATS = {
    path.name: {
        'encoding': 'utf-8-sig',
        'sep': ',',
        'date_format': '%d/%m/%y',
        'thousands_separator': None,
    }
    for path in INPUT_FILES[:-1]
}

SOURCE_FORMATS['data-2026.csv'] = {
    'encoding': 'utf-16',
    'sep': '\t',
    'date_format': '%d/%m/%Y',
    'thousands_separator': '.',
}

print(f"Folder input : {DATA_DIR}")
print(f"Folder output: {OUTPUT_DIR}")
print(f"Jumlah file yang diharapkan: {len(INPUT_FILES)}")

Folder input : C:\TUGAS\Semester5\BANJUT\PanganPulse\etl\data\dataset
Folder output: C:\TUGAS\Semester5\BANJUT\PanganPulse\etl\data\processed
Jumlah file yang diharapkan: 13


## 2. Periksa kelengkapan dan struktur file sumber

Tahap ini memeriksa apakah seluruh file tersedia, kolom identitas wajib ditemukan, dan nama kolom tanggal dapat dibaca.

In [26]:
def read_source(path, **kwargs):
    """Membaca file sumber sesuai encoding dan delimiter masing-masing."""
    settings = SOURCE_FORMATS[path.name]
    return pd.read_csv(
        path,
        sep=settings['sep'],
        encoding=settings['encoding'],
        dtype='string',
        **kwargs,
    )


def inspect_dataset():
    missing_files = [path.name for path in INPUT_FILES if not path.is_file()]
    if missing_files:
        raise FileNotFoundError(
            f"File sumber belum lengkap: {missing_files}"
        )

    rows = []

    for path in INPUT_FILES:
        settings = SOURCE_FORMATS[path.name]
        raw_columns = [
            column.strip()
            for column in read_source(path, nrows=0).columns
        ]

        normalized_columns = {
            COLUMNS_RENAME.get(column, column)
            for column in raw_columns
        }

        missing_columns = [
            column for column in REQUIRED_COLUMNS
            if column not in normalized_columns
        ]

        if missing_columns:
            raise ValueError(
                f"{path.name}: kolom wajib tidak ditemukan: {missing_columns}"
            )

        date_columns = [
            column for column in raw_columns
            if column not in COLUMNS_RENAME
            and column not in ['No', 'HET/HA']
        ]

        parsed_dates = pd.to_datetime(
            date_columns,
            format=settings['date_format'],
            errors='coerce'
        )

        if not date_columns or parsed_dates.isna().any():
            raise ValueError(
                f"{path.name}: terdapat nama kolom tanggal yang tidak valid."
            )

        row_count = sum(
            len(chunk)
            for chunk in read_source(path, chunksize=50_000)
        )

        rows.append({
            'file': path.name,
            'jumlah_baris_sumber': row_count,
            'jumlah_kolom_tanggal': len(date_columns),
            'tanggal_awal': parsed_dates.min().date(),
            'tanggal_akhir': parsed_dates.max().date(),
        })

    return pd.DataFrame(rows)


dataset_overview = inspect_dataset()
display(dataset_overview)
print(f"Total baris sumber: {dataset_overview['jumlah_baris_sumber'].sum():,}")

dataset_overview.to_csv(
    OUTPUT_DIR / 'dataset_overview.csv',
    index=False,
    encoding='utf-8-sig'
)

,file,jumlah_baris_sumber,jumlah_kolom_tanggal,tanggal_awal,tanggal_akhir
0,januari.csv,12479,19,2025-01-02,2025-01-31
1,februari.csv,12518,20,2025-02-03,2025-02-28
2,maret.csv,12521,19,2025-03-03,2025-03-27
3,april.csv,12473,16,2025-04-08,2025-04-30
4,mei.csv,12473,17,2025-05-02,2025-05-28
5,juni.csv,12494,18,2025-06-02,2025-06-30
6,juli.csv,12515,23,2025-07-01,2025-07-31
7,agustus.csv,12518,20,2025-08-01,2025-08-29
8,september.csv,12548,21,2025-09-01,2025-09-30
9,oktober.csv,12549,23,2025-10-01,2025-10-31


Total baris sumber: 158,051


## 3. Definisikan proses transformasi dan pembersihan

Data sumber berbentuk wide (kolom tanggal tersebar ke samping). Fungsi berikut mengubahnya menjadi long: satu baris untuk satu wilayah, komoditas, dan tanggal.

In [27]:
def normalize_code(values):
    """Menjaga kode sebagai teks dan menghapus akhiran .0 jika ada."""
    return (
        values.astype('string')
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
    )


def transform_source(path):
    settings = SOURCE_FORMATS[path.name]
    raw = read_source(path)
    raw.columns = [column.strip() for column in raw.columns]

    date_columns = [
        column for column in raw.columns
        if column not in COLUMNS_RENAME
        and column not in ['No', 'HET/HA']
    ]

    data = raw.rename(columns=COLUMNS_RENAME)
    data = data.drop(
        columns=[column for column in ['No', 'HET/HA'] if column in data],
        errors='ignore'
    ).copy()

    # Tambahkan kolom identitas yang tidak tersedia pada sumber tertentu.
    for column in IDENTIFIER_COLUMNS:
        if column not in data.columns:
            data[column] = pd.NA

    for column in ['kode_provinsi', 'kode_daerah', 'kode_komoditas']:
        data[column] = normalize_code(data[column])

    for column in ['provinsi', 'kabupaten_kota']:
        data[column] = data[column].astype('string').str.strip()

    # Normalisasi dasar nama, belum menyamakan alias antar-tahun.
    data['komoditas'] = (
        data['komoditas']
        .astype('string')
        .str.strip()
        .str.upper()
    )

    long = data.melt(
        id_vars=IDENTIFIER_COLUMNS,
        value_vars=date_columns,
        var_name='tanggal_sumber',
        value_name='harga_sumber',
    )

    long['tanggal'] = pd.to_datetime(
        long['tanggal_sumber'],
        format=settings['date_format'],
        errors='coerce'
    )

    price_values = long['harga_sumber'].astype('string').str.strip()

    if settings['thousands_separator']:
        price_values = price_values.str.replace(
            settings['thousands_separator'],
            '',
            regex=False
        )

    long['harga'] = pd.to_numeric(price_values, errors='coerce')

    core_identifiers = [
        'kode_daerah', 'provinsi', 'kabupaten_kota', 'komoditas'
    ]

    has_identifiers = (
        long[core_identifiers].notna().all(axis=1)
        & long[core_identifiers].ne('').all(axis=1)
    )

    valid = (
        has_identifiers
        & long['tanggal'].notna()
        & long['harga'].gt(0)
    )

    cleaned = long.loc[
        valid,
        IDENTIFIER_COLUMNS + ['tanggal', 'harga']
    ].copy()

    duplicate_mask = cleaned.duplicated(
        subset=['kode_daerah', 'komoditas', 'tanggal'],
        keep='last'
    )

    duplicate_count = int(duplicate_mask.sum())
    cleaned = cleaned.loc[~duplicate_mask].copy()

    cleaned = cleaned.sort_values(
        ['tanggal', 'kode_daerah', 'komoditas']
    ).reset_index(drop=True)

    quality = {
        'file_sumber': path.name,
        'baris_kandidat': len(long),
        'harga_kosong_atau_non_numerik': int(long['harga'].isna().sum()),
        'harga_nol_atau_negatif': int(
            (long['harga'].notna() & long['harga'].le(0)).sum()
        ),
        'tanggal_tidak_valid': int(long['tanggal'].isna().sum()),
        'baris_tanpa_identitas': int((~has_identifiers).sum()),
        'duplikat_dihapus': duplicate_count,
        'baris_diterima': len(cleaned),
        'baris_ditolak': len(long) - len(cleaned),
    }

    return cleaned, quality


print("Fungsi transformasi siap.")

Fungsi transformasi siap.


## 4. Jalankan ETL untuk seluruh file sumber

Setiap file sumber diproses secara terpisah. Hasilnya disimpan ke folder `processed` dan ringkasan kualitas disimpan pada `etl_log.csv`. Jalankan tahap ini setelah pemeriksaan file berhasil.

In [28]:
def run_etl():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summaries = []

    for path in INPUT_FILES:
        cleaned, quality = transform_source(path)
        output_path = OUTPUT_DIR / f'{path.stem}_cleaned.csv'

        cleaned.to_csv(
            output_path,
            index=False,
            date_format='%Y-%m-%d',
            encoding='utf-8-sig'
        )

        summaries.append({
            **quality,
            'file_hasil': output_path.name,
        })

    summary = pd.DataFrame(summaries)
    summary.to_csv(
        OUTPUT_DIR / 'etl_log.csv',
        index=False,
        encoding='utf-8-sig'
    )

    return summary


etl_summary = run_etl()
display(etl_summary)

print(f"Total observasi diterima: {etl_summary['baris_diterima'].sum():,}")
print(f"Total observasi ditolak : {etl_summary['baris_ditolak'].sum():,}")
print(f"Hasil ETL disimpan di   : {OUTPUT_DIR}")

,file_sumber,baris_kandidat,harga_kosong_atau_non_numerik,harga_nol_atau_negatif,tanggal_tidak_valid,baris_tanpa_identitas,duplikat_dihapus,baris_diterima,baris_ditolak,file_hasil
0,januari.csv,237101,317,0,0,19,0,236784,317,januari_cleaned.csv
1,februari.csv,250360,1637,0,0,20,0,248723,1637,februari_cleaned.csv
2,maret.csv,237899,1593,0,0,19,0,236306,1593,maret_cleaned.csv
3,april.csv,199568,562,0,0,16,0,199006,562,april_cleaned.csv
4,mei.csv,212041,490,0,0,17,0,211551,490,mei_cleaned.csv
5,juni.csv,224892,832,0,0,18,0,224060,832,juni_cleaned.csv
6,juli.csv,287845,726,0,0,23,0,287119,726,juli_cleaned.csv
7,agustus.csv,250360,213,0,0,20,0,250147,213,agustus_cleaned.csv
8,september.csv,263508,339,0,0,21,0,263169,339,september_cleaned.csv
9,oktober.csv,288627,51,0,0,0,0,288576,51,oktober_cleaned.csv


Total observasi diterima: 4,595,223
Total observasi ditolak : 27,001
Hasil ETL disimpan di   : C:\TUGAS\Semester5\BANJUT\PanganPulse\etl\data\processed


## 5. Muat hasil ETL dan gabungkan 2025–2026

**Urutan penting:** `prices_all` dibuat terlebih dahulu dari seluruh file `*_cleaned.csv`. Setelah itu barulah dataset gabungan disimpan dan dipakai pada analisis berikutnya.

In [29]:
CLEANED_FILES = [
    OUTPUT_DIR / f'{path.stem}_cleaned.csv'
    for path in INPUT_FILES
]

missing_cleaned_files = [
    path.name for path in CLEANED_FILES if not path.is_file()
]

if missing_cleaned_files:
    raise FileNotFoundError(
        f"Hasil ETL belum tersedia: {missing_cleaned_files}. Jalankan bagian 4 terlebih dahulu."
    )

CODE_DTYPES = {
    'kode_provinsi': 'string',
    'kode_daerah': 'string',
    'kode_komoditas': 'string',
    'provinsi': 'string',
    'kabupaten_kota': 'string',
    'komoditas': 'string',
}

cleaned_parts = [
    pd.read_csv(
        path,
        dtype=CODE_DTYPES,
        parse_dates=['tanggal']
    )
    for path in CLEANED_FILES
]

prices_all = pd.concat(cleaned_parts, ignore_index=True)

# Periksa duplikasi lintas file, misalnya jika dua file memiliki tanggal yang tumpang tindih.
GLOBAL_KEY = ['kode_daerah', 'komoditas', 'tanggal']
global_duplicate_count = int(prices_all.duplicated(GLOBAL_KEY).sum())

if global_duplicate_count > 0:
    prices_all = prices_all.drop_duplicates(
        subset=GLOBAL_KEY,
        keep='last'
    ).copy()

prices_all['tanggal'] = pd.to_datetime(
    prices_all['tanggal'],
    errors='coerce'
)
prices_all['tahun'] = prices_all['tanggal'].dt.year
prices_all = prices_all.sort_values(
    ['tanggal', 'kode_daerah', 'komoditas']
).reset_index(drop=True)

print("Jumlah file hasil ETL:", len(CLEANED_FILES))
print("Duplikasi lintas file yang dihapus:", global_duplicate_count)
print("Total observasi gabungan:", f"{len(prices_all):,}")
display(prices_all.head())

# Simpan dataset gabungan setelah prices_all benar-benar dibuat.
combined_output = OUTPUT_DIR / 'harga_pangan_2025_2026.csv'
prices_all.to_csv(
    combined_output,
    index=False,
    date_format='%Y-%m-%d',
    encoding='utf-8-sig'
)

print("Dataset gabungan disimpan:", combined_output)

Jumlah file hasil ETL: 13
Duplikasi lintas file yang dihapus: 0
Total observasi gabungan: 4,595,223


,kode_provinsi,provinsi,kode_daerah,kabupaten_kota,kode_komoditas,komoditas,tanggal,harga,tahun
0,11,Aceh,1101,Kab. Simeulue,117137,BAWANG MERAH,2025-01-02,45000.0,2025
1,11,Aceh,1101,Kab. Simeulue,117138,BAWANG PUTIH,2025-01-02,45000.0,2025
2,11,Aceh,1101,Kab. Simeulue,111001002,BERAS MEDIUM,2025-01-02,13450.0,2025
3,11,Aceh,1101,Kab. Simeulue,111001001,BERAS PREMIUM,2025-01-02,15000.0,2025
4,11,Aceh,1101,Kab. Simeulue,117104,CABAI MERAH,2025-01-02,60000.0,2025


Dataset gabungan disimpan: C:\TUGAS\Semester5\BANJUT\PanganPulse\etl\data\processed\harga_pangan_2025_2026.csv


## 6. Ringkasan kualitas dan cakupan data

Periksa jumlah observasi, wilayah, komoditas, dan rentang tanggal per tahun sebelum menyusun mapping.

In [30]:
yearly_summary = (
    prices_all.groupby('tahun')
    .agg(
        jumlah_observasi=('harga', 'count'),
        jumlah_wilayah=('kode_daerah', 'nunique'),
        jumlah_komoditas=('komoditas', 'nunique'),
        tanggal_awal=('tanggal', 'min'),
        tanggal_akhir=('tanggal', 'max'),
    )
)

display(yearly_summary)

print("Total wilayah gabungan:", prices_all['kode_daerah'].nunique())
print("Total nama komoditas gabungan:", prices_all['komoditas'].nunique())
print("Missing value per kolom:")
display(prices_all.isna().sum().to_frame('jumlah_missing'))

# Ringkasan kualitas harga setelah transformasi.
display(prices_all['harga'].describe())

,jumlah_observasi,jumlah_wilayah,jumlah_komoditas,tanggal_awal,tanggal_akhir
tahun,,,,,
2025,2959822,514,27,2025-01-02,2025-12-31
2026,1635401,514,17,2026-01-02,2026-11-13


Total wilayah gabungan: 544
Total nama komoditas gabungan: 38
Missing value per kolom:


,jumlah_missing
kode_provinsi,1635401
provinsi,0
kode_daerah,0
kabupaten_kota,0
kode_komoditas,1635401
komoditas,0
tanggal,0
harga,0
tahun,0


count    4.595223e+06
mean     3.249546e+04
std      3.073870e+04
min      1.000000e+03
25%      1.400000e+04
50%      2.000000e+04
75%      4.000000e+04
max      2.433330e+05
Name: harga, dtype: float64

## 7. Bandingkan daftar komoditas 2025 dan 2026

Pada tahap ini nama hanya dibandingkan secara persis setelah `strip()` dan `upper()`. Nama yang mirip tetapi berbeda tetap dianggap sebagai nama berbeda.

In [31]:
commodities_2025 = set(
    prices_all.loc[
        prices_all['tahun'] == 2025, 'komoditas'
    ].dropna().unique()
)

commodities_2026 = set(
    prices_all.loc[
        prices_all['tahun'] == 2026, 'komoditas'
    ].dropna().unique()
)

same_name_commodities = commodities_2025 & commodities_2026
only_2025 = commodities_2025 - commodities_2026
only_2026 = commodities_2026 - commodities_2025
all_original_names = commodities_2025 | commodities_2026

print("Jumlah nama komoditas 2025:", len(commodities_2025))
print("Jumlah nama komoditas 2026:", len(commodities_2026))
print("Nama yang sama persis      :", len(same_name_commodities))
print("Hanya muncul pada 2025     :", len(only_2025))
print("Hanya muncul pada 2026     :", len(only_2026))
print("Total nama unik sebelum mapping:", len(all_original_names))

print("\nNama yang sama persis di kedua tahun:")
print(sorted(same_name_commodities))

print("\nHanya ada di 2025:")
print(sorted(only_2025))

print("\nHanya ada di 2026:")
print(sorted(only_2026))

Jumlah nama komoditas 2025: 27
Jumlah nama komoditas 2026: 17
Nama yang sama persis      : 6
Hanya muncul pada 2025     : 21
Hanya muncul pada 2026     : 11
Total nama unik sebelum mapping: 38

Nama yang sama persis di kedua tahun:
['BAWANG MERAH', 'BERAS MEDIUM', 'BERAS PREMIUM', 'DAGING AYAM RAS', 'TELUR AYAM RAS', 'TEPUNG TERIGU']

Hanya ada di 2025:
['BAWANG PUTIH', 'CABAI MERAH', 'CABAI RAWIT', 'DAGING SAPI', 'GULA PASIR', 'IKAN KEMBUNG/IKAN GEMBUNG/ IKAN BANYAR/IKAN GEMBOLO/ IKAN ASO-ASO', 'JERUK', 'KACANG PANJANG', 'KANGKUNG', 'KENTANG', 'KETIMUN', 'MIE KERING INSTANT', 'MINYAK GORENG', 'PISANG', 'SAWI HIJAU', 'SUSU BUBUK', 'SUSU BUBUK UNTUK BALITA', 'TAHU MENTAH', 'TEMPE', 'TOMAT', 'UDANG BASAH']

Hanya ada di 2026:
['BAWANG PUTIH HONAN', 'CABAI MERAH BESAR', 'CABAI MERAH KERITING', 'CABAI RAWIT MERAH', 'DAGING SAPI PAHA BELAKANG', 'GARAM HALUS', 'GULA PASIR CURAH', 'IKAN KEMBUNG', 'MINYAK GORENG SAWIT CURAH', 'MINYAK GORENG SAWIT KEMASAN PREMIUM', 'MINYAKITA']


## 8. Susun kandidat mapping nama komoditas

Daftar berikut adalah **kandidat untuk diperiksa**, bukan mapping final. Beberapa nama 2026 lebih spesifik daripada nama 2025. Satu nama umum seperti `CABAI MERAH` atau `MINYAK GORENG` juga memiliki lebih dari satu kandidat, sehingga tidak boleh langsung digabungkan semuanya.

In [32]:
mapping_candidates = [
    {
        'komoditas_2025': 'BAWANG PUTIH',
        'kandidat_2026': 'BAWANG PUTIH HONAN',
        'standar_kandidat': 'BAWANG PUTIH',
        'catatan': 'Perlu memastikan apakah jenis Honan sebanding dengan kategori umum.'
    },
    {
        'komoditas_2025': 'CABAI MERAH',
        'kandidat_2026': 'CABAI MERAH BESAR',
        'standar_kandidat': 'CABAI MERAH',
        'catatan': 'Kandidat A; kategori 2026 lebih spesifik.'
    },
    {
        'komoditas_2025': 'CABAI MERAH',
        'kandidat_2026': 'CABAI MERAH KERITING',
        'standar_kandidat': 'CABAI MERAH',
        'catatan': 'Kandidat B; perlu dibandingkan dengan kandidat A.'
    },
    {
        'komoditas_2025': 'CABAI RAWIT',
        'kandidat_2026': 'CABAI RAWIT MERAH',
        'standar_kandidat': 'CABAI RAWIT',
        'catatan': 'Perlu memastikan kategori 2025 tidak mencakup jenis rawit lain.'
    },
    {
        'komoditas_2025': 'DAGING SAPI',
        'kandidat_2026': 'DAGING SAPI PAHA BELAKANG',
        'standar_kandidat': 'DAGING SAPI',
        'catatan': 'Kategori 2026 menyebut potongan tertentu.'
    },
    {
        'komoditas_2025': 'GULA PASIR',
        'kandidat_2026': 'GULA PASIR CURAH',
        'standar_kandidat': 'GULA PASIR',
        'catatan': 'Perlu memastikan kategori 2025 juga merujuk gula curah.'
    },
    {
        'komoditas_2025': 'MINYAK GORENG',
        'kandidat_2026': 'MINYAK GORENG SAWIT CURAH',
        'standar_kandidat': 'MINYAK GORENG',
        'catatan': 'Kandidat minyak goreng curah.'
    },
    {
        'komoditas_2025': 'MINYAK GORENG',
        'kandidat_2026': 'MINYAK GORENG SAWIT KEMASAN PREMIUM',
        'standar_kandidat': 'MINYAK GORENG',
        'catatan': 'Kandidat minyak goreng kemasan premium.'
    },
    {
        'komoditas_2025': 'MINYAK GORENG',
        'kandidat_2026': 'MINYAKITA',
        'standar_kandidat': 'MINYAK GORENG',
        'catatan': 'Perlu kehati-hatian karena Minyakita merupakan kategori produk tersendiri.'
    },
]

candidate_check = pd.DataFrame(mapping_candidates)
candidate_check['ada_di_2025'] = candidate_check['komoditas_2025'].isin(
    commodities_2025
)
candidate_check['ada_di_2026'] = candidate_check['kandidat_2026'].isin(
    commodities_2026
)

# Status hanya menunjukkan apakah label kandidat ditemukan di dataset.
candidate_check['dapat_diuji'] = (
    candidate_check['ada_di_2025']
    & candidate_check['ada_di_2026']
)

display(candidate_check)

candidate_check.to_csv(
    OUTPUT_DIR / 'commodity_mapping_candidates.csv',
    index=False,
    encoding='utf-8-sig'
)

,komoditas_2025,kandidat_2026,standar_kandidat,catatan,ada_di_2025,ada_di_2026,dapat_diuji
0,BAWANG PUTIH,BAWANG PUTIH HONAN,BAWANG PUTIH,Perlu memastikan apakah jenis Honan sebanding ...,True,True,True
1,CABAI MERAH,CABAI MERAH BESAR,CABAI MERAH,Kandidat A; kategori 2026 lebih spesifik.,True,True,True
2,CABAI MERAH,CABAI MERAH KERITING,CABAI MERAH,Kandidat B; perlu dibandingkan dengan kandidat A.,True,True,True
3,CABAI RAWIT,CABAI RAWIT MERAH,CABAI RAWIT,Perlu memastikan kategori 2025 tidak mencakup ...,True,True,True
4,DAGING SAPI,DAGING SAPI PAHA BELAKANG,DAGING SAPI,Kategori 2026 menyebut potongan tertentu.,True,True,True
5,GULA PASIR,GULA PASIR CURAH,GULA PASIR,Perlu memastikan kategori 2025 juga merujuk gu...,True,True,True
6,MINYAK GORENG,MINYAK GORENG SAWIT CURAH,MINYAK GORENG,Kandidat minyak goreng curah.,True,True,True
7,MINYAK GORENG,MINYAK GORENG SAWIT KEMASAN PREMIUM,MINYAK GORENG,Kandidat minyak goreng kemasan premium.,True,True,True
8,MINYAK GORENG,MINYAKITA,MINYAK GORENG,Perlu kehati-hatian karena Minyakita merupakan...,True,True,True


## 9. Siapkan periode perbandingan harga

Agar perbandingan tidak berat sebelah, gunakan periode kalender yang sama pada kedua tahun. Contohnya, jika data 2026 tersedia sampai 13 November, data 2025 juga dibatasi sampai 13 November. Perbandingan dilakukan pada wilayah yang mempunyai observasi untuk kedua nama komoditas.

In [33]:
last_date_2026 = prices_all.loc[
    prices_all['tahun'] == 2026,
    'tanggal'
].max()

if pd.isna(last_date_2026):
    raise ValueError("Tidak ditemukan observasi tahun 2026.")

period_end_2026 = last_date_2026
period_end_2025 = pd.Timestamp(
    year=2025,
    month=last_date_2026.month,
    day=last_date_2026.day
)

prices_comparison = prices_all[
    (
        (prices_all['tahun'] == 2025)
        & (prices_all['tanggal'] <= period_end_2025)
    )
    |
    (
        (prices_all['tahun'] == 2026)
        & (prices_all['tanggal'] <= period_end_2026)
    )
].copy()

print("Periode 2025: 1 Januari sampai", period_end_2025.date())
print("Periode 2026: 1 Januari sampai", period_end_2026.date())
print("Jumlah observasi untuk perbandingan:", f"{len(prices_comparison):,}")

Periode 2025: 1 Januari sampai 2025-11-13
Periode 2026: 1 Januari sampai 2026-11-13
Jumlah observasi untuk perbandingan: 4,193,789


## 10. Bandingkan harga kandidat antar-tahun

Untuk setiap pasangan kandidat, hitung median harga per wilayah pada 2025 dan 2026. Setelah itu, bandingkan hanya wilayah yang muncul pada kedua kelompok data.

Indikator yang dihasilkan:
- `jumlah_wilayah_sama`: ukuran sampel wilayah yang dapat dibandingkan.
- `perubahan_median_persen`: perubahan antara median harga 2025 dan 2026 pada wilayah berpasangan.
- `median_perubahan_per_wilayah_persen`: median persentase perubahan di tingkat wilayah.
- `korelasi_spearman`: kemiripan urutan relatif harga antarwilayah.
- `p_value_wilcoxon`: uji perubahan berpasangan; nilai kecil menunjukkan perubahan harga yang terdeteksi secara statistik, bukan bukti bahwa mapping salah.

In [34]:
def compare_commodity_pair(old_name, new_name, period_data):
    old_prices = (
        period_data[
            (period_data['tahun'] == 2025)
            & (period_data['komoditas'] == old_name)
        ]
        .groupby('kode_daerah')['harga']
        .median()
        .rename('harga_2025')
    )

    new_prices = (
        period_data[
            (period_data['tahun'] == 2026)
            & (period_data['komoditas'] == new_name)
        ]
        .groupby('kode_daerah')['harga']
        .median()
        .rename('harga_2026')
    )

    paired = pd.concat(
        [old_prices, new_prices],
        axis=1,
        join='inner'
    ).dropna()

    if paired.empty:
        return {
            'komoditas_2025': old_name,
            'kandidat_2026': new_name,
            'jumlah_wilayah_sama': 0,
            'median_harga_2025': np.nan,
            'median_harga_2026': np.nan,
            'perubahan_median_persen': np.nan,
            'median_perubahan_per_wilayah_persen': np.nan,
            'wilayah_harga_naik': 0,
            'wilayah_harga_turun': 0,
            'korelasi_spearman': np.nan,
            'p_value_wilcoxon': np.nan,
        }

    paired['perubahan_persen'] = (
        paired['harga_2026'] / paired['harga_2025'] - 1
    ) * 100

    median_2025 = paired['harga_2025'].median()
    median_2026 = paired['harga_2026'].median()
    overall_change = (median_2026 / median_2025 - 1) * 100

    spearman_corr = np.nan
    wilcoxon_p = np.nan

    if SCIPY_AVAILABLE and len(paired) >= 2:
        if (
            paired['harga_2025'].nunique() > 1
            and paired['harga_2026'].nunique() > 1
        ):
            spearman_corr = float(
                spearmanr(
                    paired['harga_2025'],
                    paired['harga_2026']
                ).statistic
            )

    if SCIPY_AVAILABLE and len(paired) >= 10:
        differences = paired['harga_2026'] - paired['harga_2025']

        if np.isclose(differences, 0).all():
            wilcoxon_p = 1.0
        else:
            try:
                wilcoxon_p = float(
                    wilcoxon(
                        paired['harga_2026'],
                        paired['harga_2025'],
                        alternative='two-sided'
                    ).pvalue
                )
            except ValueError:
                wilcoxon_p = np.nan

    return {
        'komoditas_2025': old_name,
        'kandidat_2026': new_name,
        'jumlah_wilayah_sama': len(paired),
        'median_harga_2025': median_2025,
        'median_harga_2026': median_2026,
        'perubahan_median_persen': overall_change,
        'median_perubahan_per_wilayah_persen': paired['perubahan_persen'].median(),
        'wilayah_harga_naik': int((paired['perubahan_persen'] > 0).sum()),
        'wilayah_harga_turun': int((paired['perubahan_persen'] < 0).sum()),
        'korelasi_spearman': spearman_corr,
        'p_value_wilcoxon': wilcoxon_p,
    }


comparison_rows = []

for candidate in mapping_candidates:
    old_name = candidate['komoditas_2025']
    new_name = candidate['kandidat_2026']

    if old_name not in commodities_2025 or new_name not in commodities_2026:
        continue

    comparison_rows.append(
        compare_commodity_pair(
            old_name,
            new_name,
            prices_comparison
        )
    )

commodity_price_comparison = pd.DataFrame(comparison_rows)

if not commodity_price_comparison.empty:
    commodity_price_comparison = commodity_price_comparison.sort_values(
        ['komoditas_2025', 'korelasi_spearman'],
        ascending=[True, False]
    ).reset_index(drop=True)

display(commodity_price_comparison.round(3))

commodity_price_comparison.to_csv(
    OUTPUT_DIR / 'commodity_pair_price_comparison.csv',
    index=False,
    encoding='utf-8-sig'
)

,komoditas_2025,kandidat_2026,jumlah_wilayah_sama,median_harga_2025,median_harga_2026,perubahan_median_persen,median_perubahan_per_wilayah_persen,wilayah_harga_naik,wilayah_harga_turun,korelasi_spearman,p_value_wilcoxon
0,BAWANG PUTIH,BAWANG PUTIH HONAN,456,39333.330,35000.0,-11.017,-7.021,63,323,0.770,0.000
1,CABAI MERAH,CABAI MERAH KERITING,463,45000.000,41667.0,-7.407,-4.762,151,270,0.732,0.000
2,CABAI MERAH,CABAI MERAH BESAR,372,44458.333,41833.5,-5.904,-4.000,143,206,0.729,0.041
3,CABAI RAWIT,CABAI RAWIT MERAH,418,48583.335,61667.0,26.930,28.883,355,47,0.694,0.000
4,DAGING SAPI,DAGING SAPI PAHA BELAKANG,406,134500.000,140000.0,4.089,5.735,261,33,0.778,0.000
5,GULA PASIR,GULA PASIR CURAH,463,18000.000,18333.0,1.850,1.887,263,81,0.726,0.000
6,MINYAK GORENG,MINYAK GORENG SAWIT KEMASAN PREMIUM,481,18611.110,23000.0,23.582,21.857,476,4,0.635,0.000
7,MINYAK GORENG,MINYAK GORENG SAWIT CURAH,385,18344.445,18900.0,3.028,2.904,235,138,0.367,0.000
8,MINYAK GORENG,MINYAKITA,481,18600.003,15700.0,-15.591,-15.135,8,469,0.250,0.000


## 11. Panduan membaca hasil perbandingan

Gunakan indikator secara bersama-sama:

- **Jumlah wilayah sama:** semakin banyak wilayah berpasangan, semakin besar cakupan perbandingan. Pasangan dengan sampel sangat kecil perlu ditafsirkan hati-hati.
- **Perubahan harga:** kenaikan atau penurunan yang besar dapat terjadi karena inflasi, pasokan, musim, atau perbedaan kategori produk. Perubahan besar tidak otomatis berarti mapping salah.
- **Korelasi Spearman:** nilai mendekati +1 menunjukkan urutan relatif harga antarwilayah cenderung serupa. Ini dapat menjadi bukti pendukung, tetapi tidak membuktikan dua label adalah produk yang sama.
- **Uji Wilcoxon:** `p_value_wilcoxon < 0.05` menunjukkan terdapat perbedaan berpasangan yang terdeteksi secara statistik. Jangan gunakan p-value sebagai satu-satunya dasar menerima atau menolak mapping.

**Penting:** kesesuaian mapping tetap perlu mempertimbangkan definisi komoditas, spesifikasi produk, dan bila tersedia kode komoditas atau metadata resmi. Perbandingan harga hanya menjadi pemeriksaan pendukung.

## 12. Tetapkan mapping final setelah kandidat ditinjau

Isi dictionary `COMMODITY_MAPPING` hanya untuk alias yang sudah diputuskan. Kunci adalah nama asli yang ingin diubah; nilai adalah nama standar. Jangan menggabungkan kandidat alternatif sebelum dipilih. Nama yang tidak dimasukkan ke dictionary akan tetap menggunakan nama aslinya.

In [ ]:
# ==========================================================
# PERBANDINGAN HARGA DESEMBER 2025 VS JANUARI 2026
# ==========================================================

prices_monthly = prices_all.copy()

prices_monthly['tanggal'] = pd.to_datetime(
    prices_monthly['tanggal'],
    errors='coerce'
)

prices_monthly['tahun'] = prices_monthly['tanggal'].dt.year
prices_monthly['bulan'] = prices_monthly['tanggal'].dt.month

# Ambil data Desember 2025 dan Januari 2026
monthly_source = prices_monthly[
    (
        (prices_monthly['tahun'] == 2025) &
        (prices_monthly['bulan'] == 12)
    ) |
    (
        (prices_monthly['tahun'] == 2026) &
        (prices_monthly['bulan'] == 1)
    )
].copy()

# Periksa rentang tanggal yang tersedia
date_coverage = (
    monthly_source.groupby(['tahun', 'bulan'])['tanggal']
    .agg(
        tanggal_awal='min',
        tanggal_akhir='max',
        jumlah_observasi='count'
    )
    .reset_index()
)

display(date_coverage)


# ==========================================================
# HITUNG RATA-RATA HARGA PER WILAYAH
# ==========================================================

monthly_region_avg = (
    monthly_source
    .groupby(['tahun', 'bulan', 'kode_daerah', 'komoditas'])
    .agg(
        rata_rata_harga=('harga', 'mean'),
        jumlah_observasi=('harga', 'count'),
        jumlah_hari_observasi=('tanggal', 'nunique')
    )
    .reset_index()
)


# ==========================================================
# FUNGSI PERBANDINGAN DUA KOMODITAS
# ==========================================================

def compare_month_pair(old_name, new_name, jenis):

    december = (
        monthly_region_avg[
            (monthly_region_avg['tahun'] == 2025) &
            (monthly_region_avg['bulan'] == 12) &
            (monthly_region_avg['komoditas'] == old_name)
        ]
        .drop(columns=['tahun', 'bulan', 'komoditas'])
        .rename(columns={
            'rata_rata_harga': 'harga_desember',
            'jumlah_observasi': 'observasi_desember',
            'jumlah_hari_observasi': 'hari_desember'
        })
    )

    january = (
        monthly_region_avg[
            (monthly_region_avg['tahun'] == 2026) &
            (monthly_region_avg['bulan'] == 1) &
            (monthly_region_avg['komoditas'] == new_name)
        ]
        .drop(columns=['tahun', 'bulan', 'komoditas'])
        .rename(columns={
            'rata_rata_harga': 'harga_januari',
            'jumlah_observasi': 'observasi_januari',
            'jumlah_hari_observasi': 'hari_januari'
        })
    )

    # Hanya wilayah yang memiliki data di kedua bulan
    paired = december.merge(
        january,
        on='kode_daerah',
        how='inner'
    )

    if paired.empty:
        return {
            'jenis': jenis,
            'komoditas_desember_2025': old_name,
            'komoditas_januari_2026': new_name,
            'jumlah_wilayah': 0,
            'rata_rata_desember': np.nan,
            'rata_rata_januari': np.nan,
            'perubahan_rata_rata_persen': np.nan,
            'rata_rata_perubahan_wilayah_persen': np.nan,
            'median_perubahan_wilayah_persen': np.nan,
            'korelasi_spearman': np.nan,
            'wilayah_naik': 0,
            'wilayah_turun': 0,
            'wilayah_tetap': 0,
            'rata_hari_observasi_desember': np.nan,
            'rata_hari_observasi_januari': np.nan
        }

    # Perubahan harga pada setiap wilayah
    paired['perubahan_persen'] = (
        (paired['harga_januari'] /
         paired['harga_desember']) - 1
    ) * 100

    # Rata-rata harga antarwilayah.
    # Setiap wilayah memiliki bobot yang sama.
    mean_december = paired['harga_desember'].mean()
    mean_january = paired['harga_januari'].mean()

    overall_change = (
        (mean_january / mean_december) - 1
    ) * 100

    # Korelasi peringkat harga antarwilayah
    correlation = np.nan

    if (
        SCIPY_AVAILABLE
        and len(paired) >= 2
        and paired['harga_desember'].nunique() > 1
        and paired['harga_januari'].nunique() > 1
    ):
        correlation = spearmanr(
            paired['harga_desember'],
            paired['harga_januari']
        ).statistic

    return {
        'jenis': jenis,
        'komoditas_desember_2025': old_name,
        'komoditas_januari_2026': new_name,
        'jumlah_wilayah': len(paired),
        'rata_rata_desember': mean_december,
        'rata_rata_januari': mean_january,
        'perubahan_rata_rata_persen': overall_change,
        'rata_rata_perubahan_wilayah_persen':
            paired['perubahan_persen'].mean(),
        'median_perubahan_wilayah_persen':
            paired['perubahan_persen'].median(),
        'korelasi_spearman': correlation,
        'wilayah_naik':
            int((paired['perubahan_persen'] > 0).sum()),
        'wilayah_turun':
            int((paired['perubahan_persen'] < 0).sum()),
        'wilayah_tetap':
            int((paired['perubahan_persen'] == 0).sum()),
        'rata_hari_observasi_desember':
            paired['hari_desember'].mean(),
        'rata_hari_observasi_januari':
            paired['hari_januari'].mean()
    }


# ==========================================================
# 1. PERBANDINGAN KANDIDAT MAPPING
# ==========================================================

monthly_results = []

for candidate in mapping_candidates:

    monthly_results.append(
        compare_month_pair(
            candidate['komoditas_2025'],
            candidate['kandidat_2026'],
            'Kandidat mapping'
        )
    )


# ==========================================================
# 2. PERBANDINGAN KOMODITAS DENGAN NAMA SAMA
#    DIGUNAKAN SEBAGAI REFERENSI
# ==========================================================

december_names = set(
    monthly_region_avg.loc[
        (monthly_region_avg['tahun'] == 2025) &
        (monthly_region_avg['bulan'] == 12),
        'komoditas'
    ].unique()
)

january_names = set(
    monthly_region_avg.loc[
        (monthly_region_avg['tahun'] == 2026) &
        (monthly_region_avg['bulan'] == 1),
        'komoditas'
    ].unique()
)

common_monthly_commodities = sorted(
    december_names & january_names
)

for commodity in common_monthly_commodities:

    monthly_results.append(
        compare_month_pair(
            commodity,
            commodity,
            'Referensi nama sama'
        )
    )


# ==========================================================
# 3. TAMPILKAN HASIL
# ==========================================================

monthly_comparison = pd.DataFrame(monthly_results)

monthly_comparison = monthly_comparison.sort_values(
    ['jenis', 'komoditas_desember_2025']
).reset_index(drop=True)

display(monthly_comparison.round(2))


# ==========================================================
# 4. SIMPAN HASIL
# ==========================================================

monthly_comparison.to_csv(
    OUTPUT_DIR / 'commodity_monthly_price_comparison.csv',
    index=False,
    encoding='utf-8-sig'
)

print("Hasil perbandingan bulanan berhasil disimpan.")

,tahun,bulan,tanggal_awal,tanggal_akhir,jumlah_observasi
0,2025,12,2025-12-01,2025-12-31,263405
1,2026,1,2026-01-02,2026-01-30,154682


,jenis,komoditas_desember_2025,komoditas_januari_2026,jumlah_wilayah,rata_rata_desember,rata_rata_januari,perubahan_rata_rata_persen,rata_rata_perubahan_wilayah_persen,median_perubahan_wilayah_persen,korelasi_spearman,wilayah_naik,wilayah_turun,wilayah_tetap,rata_hari_observasi_desember,rata_hari_observasi_januari
0,Kandidat mapping,BAWANG PUTIH,BAWANG PUTIH HONAN,454,37451.90,38087.91,1.70,1.91,0.21,0.80,234,178,42,20.98,19.99
1,Kandidat mapping,CABAI MERAH,CABAI MERAH BESAR,365,53194.86,39673.86,-25.42,-25.27,-32.12,0.61,48,314,3,20.99,19.82
2,Kandidat mapping,CABAI MERAH,CABAI MERAH KERITING,457,54776.04,39621.20,-27.67,-26.79,-32.90,0.59,54,400,3,20.99,19.98
3,Kandidat mapping,CABAI RAWIT,CABAI RAWIT MERAH,415,64158.10,53815.39,-16.12,-14.36,-21.25,0.66,95,320,0,20.96,20.00
4,Kandidat mapping,DAGING SAPI,DAGING SAPI PAHA BELAKANG,402,134922.10,135630.00,0.52,0.72,0.00,0.86,106,79,217,20.98,20.00
5,Kandidat mapping,GULA PASIR,GULA PASIR CURAH,461,17795.70,17969.67,0.98,1.03,0.00,0.84,188,87,186,20.97,20.00
6,Kandidat mapping,MINYAK GORENG,MINYAK GORENG SAWIT CURAH,383,18698.90,17787.22,-4.88,-4.73,-4.66,0.53,84,293,6,20.99,19.84
7,Kandidat mapping,MINYAK GORENG,MINYAK GORENG SAWIT KEMASAN PREMIUM,478,19059.04,21773.20,14.24,14.26,13.78,0.74,460,15,3,20.99,20.00
8,Kandidat mapping,MINYAK GORENG,MINYAKITA,468,19005.44,17139.16,-9.82,-9.76,-10.50,0.59,16,445,7,20.99,19.96
9,Referensi nama sama,BAWANG MERAH,BAWANG MERAH,480,46066.13,42275.30,-8.23,-7.96,-10.62,0.70,106,370,4,20.99,20.00


Hasil perbandingan bulanan berhasil disimpan.


In [41]:
COMMODITY_MAPPING = {
    # Bawang
    'BAWANG PUTIH HONAN': 'BAWANG PUTIH',

    # Cabai
    'CABAI MERAH BESAR': 'CABAI MERAH',
    'CABAI RAWIT MERAH': 'CABAI RAWIT',

    # Daging
    'DAGING SAPI PAHA BELAKANG': 'DAGING SAPI',

    # Gula
    'GULA PASIR CURAH': 'GULA PASIR',

    # Minyak goreng
    'MINYAK GORENG SAWIT CURAH': 'MINYAK GORENG',
}

# Validasi: semua nama sumber yang dipetakan harus benar-benar ada di data.
unknown_mapping_keys = set(COMMODITY_MAPPING) - set(prices_all['komoditas'].dropna().unique())

if unknown_mapping_keys:
    raise ValueError(
        f"Nama komoditas pada mapping tidak ditemukan di data: {sorted(unknown_mapping_keys)}"
    )

prices_normalized = prices_all.copy()
prices_normalized['komoditas_normalisasi'] = (
    prices_normalized['komoditas']
    .map(COMMODITY_MAPPING)
    .fillna(prices_normalized['komoditas'])
)

original_name_count = prices_normalized['komoditas'].nunique()
normalized_name_count = prices_normalized['komoditas_normalisasi'].nunique()

print("Jumlah nama sebelum mapping:", original_name_count)
print("Jumlah komoditas setelah mapping:", normalized_name_count)
print("Nama yang berhasil digabung:", original_name_count - normalized_name_count)

# Tampilkan nama asli yang masuk ke masing-masing nama standar.
mapping_audit = (
    prices_normalized.groupby('komoditas_normalisasi')
    .agg(
        nama_asli=('komoditas', lambda values: sorted(values.dropna().unique().tolist())),
        jumlah_nama_asli=('komoditas', 'nunique'),
        jumlah_observasi=('harga', 'count'),
        tahun_tersedia=('tahun', lambda values: sorted(values.dropna().unique().tolist())),
    )
    .sort_index()
)

display(mapping_audit)

# Lihat cakupan setiap komoditas standar pada masing-masing tahun.
commodity_year_coverage = (
    prices_normalized.groupby(['komoditas_normalisasi', 'tahun'])
    .size()
    .unstack(fill_value=0)
)

display(commodity_year_coverage)

prices_normalized.to_csv(
    OUTPUT_DIR / 'harga_pangan_2025_2026_normalized.csv',
    index=False,
    date_format='%Y-%m-%d',
    encoding='utf-8-sig'
)

mapping_audit.to_csv(
    OUTPUT_DIR / 'commodity_mapping_audit.csv',
    encoding='utf-8-sig'
)

commodity_year_coverage.to_csv(
    OUTPUT_DIR / 'commodity_year_coverage.csv',
    encoding='utf-8-sig'
)

Jumlah nama sebelum mapping: 38
Jumlah komoditas setelah mapping: 32
Nama yang berhasil digabung: 6


,nama_asli,jumlah_nama_asli,jumlah_observasi,tahun_tersedia
komoditas_normalisasi,,,,
BAWANG MERAH,[BAWANG MERAH],1,228810,"[2025, 2026]"
BAWANG PUTIH,"[BAWANG PUTIH, BAWANG PUTIH HONAN]",2,222398,"[2025, 2026]"
BERAS MEDIUM,[BERAS MEDIUM],1,209182,"[2025, 2026]"
BERAS PREMIUM,[BERAS PREMIUM],1,214207,"[2025, 2026]"
CABAI MERAH,"[CABAI MERAH, CABAI MERAH BESAR]",2,200809,"[2025, 2026]"
CABAI MERAH KERITING,[CABAI MERAH KERITING],1,103035,[2026]
CABAI RAWIT,"[CABAI RAWIT, CABAI RAWIT MERAH]",2,205056,"[2025, 2026]"
DAGING AYAM RAS,[DAGING AYAM RAS],1,227699,"[2025, 2026]"
DAGING SAPI,"[DAGING SAPI, DAGING SAPI PAHA BELAKANG]",2,197710,"[2025, 2026]"


tahun,2025,2026
komoditas_normalisasi,,
BAWANG MERAH,120880,107930
BAWANG PUTIH,120341,102057
BERAS MEDIUM,111125,98057
BERAS PREMIUM,113431,100776
CABAI MERAH,120879,79930
CABAI MERAH KERITING,0,103035
CABAI RAWIT,108470,96586
DAGING AYAM RAS,120042,107657
DAGING SAPI,104666,93044


## 13. Bentuk matriks awal untuk clustering

Matriks menggunakan median harga per wilayah dan nama komoditas yang sudah dinormalisasi. Karena periode menggabungkan 2025 dan 2026, nilai ini merepresentasikan median selama periode pengamatan, bukan harga terkini.

Untuk saat ini tidak dilakukan `ffill` atau imputasi otomatis. Missing value diperiksa terlebih dahulu agar kita mengetahui apakah pasangan wilayah–komoditas memang tidak memiliki observasi.

In [42]:
# Jangan keluarkan komoditas secara otomatis. Tambahkan nama ke daftar ini
# hanya setelah alasan pengecualian disepakati.
EXCLUDED_COMMODITIES = []

clustering_source = prices_normalized[
    ~prices_normalized['komoditas_normalisasi'].isin(EXCLUDED_COMMODITIES)
].copy()

clustering_data = clustering_source.pivot_table(
    index='kode_daerah',
    columns='komoditas_normalisasi',
    values='harga',
    aggfunc='median'
)

print("Ukuran matriks wilayah × komoditas:", clustering_data.shape)
print("Total missing value:", int(clustering_data.isna().sum().sum()))
print(
    "Persentase missing value:",
    round(clustering_data.isna().mean().mean() * 100, 2),
    "%"
)

display(clustering_data.head())

clustering_data.to_csv(
    OUTPUT_DIR / 'clustering_matrix_before_imputation.csv',
    encoding='utf-8-sig'
)

Ukuran matriks wilayah × komoditas: (544, 32)
Total missing value: 2270
Persentase missing value: 13.04 %


komoditas_normalisasi,BAWANG MERAH,BAWANG PUTIH,BERAS MEDIUM,BERAS PREMIUM,CABAI MERAH,CABAI MERAH KERITING,CABAI RAWIT,DAGING AYAM RAS,DAGING SAPI,GARAM HALUS,...,PISANG,SAWI HIJAU,SUSU BUBUK,SUSU BUBUK UNTUK BALITA,TAHU MENTAH,TELUR AYAM RAS,TEMPE,TEPUNG TERIGU,TOMAT,UDANG BASAH
kode_daerah,,,,,,,,,,,,,,,,,,,,,
1101,40000.0,40000.0,14350.0,16925.0,60000.0,40000.0,NaN,43000.0,150000.0,13000.0,...,NaN,15000.0,NaN,NaN,25000.0,29500.0,NaN,14000.0,14000.0,100000.0
1102,37500.0,40000.0,13600.0,14600.0,32500.0,38000.0,40000.0,37000.0,150000.0,15000.0,...,15000.0,10000.0,52900.0,43900.0,13000.0,27700.0,15000.0,14500.0,8000.0,65000.0
1103,35000.0,40000.0,14500.0,16500.0,45000.0,35000.0,45000.0,32000.0,170000.0,11000.0,...,7000.0,10000.0,54000.0,45000.0,10000.0,28500.0,14000.0,14000.0,10000.0,75000.0
1104,38000.0,40000.0,14000.0,15750.0,38000.0,40000.0,40000.0,35000.0,150000.0,12000.0,...,12000.0,10000.0,NaN,42000.0,10000.0,27000.0,15000.0,13000.0,18000.0,80000.0
1105,38000.0,40000.0,14000.0,15300.0,44250.0,40000.0,45000.0,30000.0,150000.0,14000.0,...,18000.0,10000.0,44000.0,42000.0,15000.0,26700.0,16000.0,12000.0,12000.0,70000.0


## 14. Analisis missing value pada matriks clustering

Bagian ini memisahkan missing value pada matriks menjadi kombinasi wilayah–komoditas yang tidak memiliki observasi sama sekali pada periode gabungan. Informasi ini akan menjadi dasar pemilihan metode penanganan missing value.

In [43]:
missing_by_commodity = pd.DataFrame({
    'jumlah_missing': clustering_data.isna().sum(),
    'persentase_missing': clustering_data.isna().mean() * 100,
}).sort_values('persentase_missing', ascending=False)

display(missing_by_commodity.round(2))

missing_per_region = clustering_data.isna().sum(axis=1)
missing_distribution = (
    missing_per_region.value_counts()
    .sort_index()
    .rename_axis('jumlah_komoditas_missing')
    .reset_index(name='jumlah_wilayah')
)
missing_distribution['persentase_wilayah'] = (
    missing_distribution['jumlah_wilayah']
    / len(clustering_data) * 100
)

display(missing_distribution.round(2))

print("Wilayah tanpa missing:", int((missing_per_region == 0).sum()))
print("Wilayah dengan maksimal 2 missing:", int((missing_per_region <= 2).sum()))
print("Wilayah dengan lebih dari 3 missing:", int((missing_per_region > 3).sum()))

,jumlah_missing,persentase_missing
komoditas_normalisasi,,
IKAN KEMBUNG/IKAN GEMBUNG/ IKAN BANYAR/IKAN GEMBOLO/ IKAN ASO-ASO,430,79.04
IKAN KEMBUNG,417,76.65
SUSU BUBUK UNTUK BALITA,186,34.19
SUSU BUBUK,161,29.60
UDANG BASAH,99,18.20
JERUK,87,15.99
PISANG,73,13.42
TEMPE,59,10.85
DAGING SAPI,57,10.48


,jumlah_komoditas_missing,jumlah_wilayah,persentase_wilayah
0,0,33,6.07
1,1,36,6.62
2,2,184,33.82
3,3,84,15.44
4,4,70,12.87
5,5,26,4.78
6,6,20,3.68
7,7,18,3.31
8,8,14,2.57
9,9,11,2.02


Wilayah tanpa missing: 33
Wilayah dengan maksimal 2 missing: 253
Wilayah dengan lebih dari 3 missing: 207


In [44]:
regions = clustering_source['kode_daerah'].dropna().unique()
commodities = (
    clustering_source['komoditas_normalisasi']
    .dropna()
    .unique()
)

observed_pairs = (
    clustering_source.groupby(
        ['kode_daerah', 'komoditas_normalisasi']
    ).size().index
)

all_pairs = pd.MultiIndex.from_product(
    [regions, commodities],
    names=['kode_daerah', 'komoditas_normalisasi']
)

missing_pairs = all_pairs.difference(observed_pairs)
missing_pairs_df = missing_pairs.to_frame(index=False)

print("Total kombinasi wilayah × komoditas:", len(all_pairs))
print("Kombinasi tanpa observasi:", len(missing_pairs_df))

display(missing_pairs_df.head(25))

missing_pairs_df.to_csv(
    OUTPUT_DIR / 'clustering_pairs_without_observations.csv',
    index=False,
    encoding='utf-8-sig'
)

Total kombinasi wilayah × komoditas: 17408
Kombinasi tanpa observasi: 2270


,kode_daerah,komoditas_normalisasi
0,1101,CABAI RAWIT
1,1101,IKAN KEMBUNG/IKAN GEMBUNG/ IKAN BANYAR/IKAN GE...
2,1101,JERUK
3,1101,PISANG
4,1101,SUSU BUBUK
5,1101,SUSU BUBUK UNTUK BALITA
6,1101,TEMPE
7,1102,IKAN KEMBUNG
8,1102,IKAN KEMBUNG/IKAN GEMBUNG/ IKAN BANYAR/IKAN GE...
9,1103,IKAN KEMBUNG


## 14. Daily Forward Fill dan Pembentukan Matriks Clustering


In [47]:

# ==========================================================
# 1. VALIDASI DATA HASIL NORMALISASI
# ==========================================================

EXCLUDED_COMMODITIES = []

clustering_source_ffill = prices_normalized[
    ~prices_normalized['komoditas_normalisasi'].isin(
        EXCLUDED_COMMODITIES
    )
].copy()

n_commodities = (
    clustering_source_ffill['komoditas_normalisasi'].nunique()
)

print("Jumlah komoditas:", n_commodities)

if n_commodities != 32:
    raise ValueError(
        f"Diharapkan 32 komoditas, tetapi ditemukan {n_commodities}. "
        "Pastikan Cell 23 dan Cell 25 sudah dijalankan ulang setelah mapping."
    )


# ==========================================================
# 2. SIAPKAN DATA HARGA
# ==========================================================

daily_source = clustering_source_ffill[
    [
        'kode_daerah',
        'komoditas_normalisasi',
        'tanggal',
        'harga'
    ]
].copy()

daily_source['tanggal'] = pd.to_datetime(
    daily_source['tanggal'],
    errors='coerce'
)

daily_source['harga'] = pd.to_numeric(
    daily_source['harga'],
    errors='coerce'
)

daily_source = daily_source.dropna(
    subset=[
        'kode_daerah',
        'komoditas_normalisasi',
        'tanggal',
        'harga'
    ]
)

daily_source = daily_source[
    daily_source['harga'] > 0
].copy()

group_cols = [
    'kode_daerah',
    'komoditas_normalisasi'
]


# ==========================================================
# 3. AGREGASI OBSERVASI PADA TANGGAL YANG SAMA
# ==========================================================
# Jika terdapat lebih dari satu observasi untuk kombinasi
# wilayah, komoditas, dan tanggal yang sama, gunakan median.

daily_observed = (
    daily_source
    .groupby(
        group_cols + ['tanggal'],
        as_index=False,
        observed=True
    )
    .agg(
        harga_observasi=('harga', 'median')
    )
)

print("Jumlah baris sumber:", f"{len(daily_source):,}")
print("Jumlah observasi harian:", f"{len(daily_observed):,}")



# ==========================================================
# 4. BUAT DAILY TIME SERIES UNTUK SETIAP PASANGAN
# ==========================================================
# Setiap pasangan wilayah-komoditas menggunakan rentang
# tanggal observasinya sendiri.
# Tambahkan 7 hari setelah observasi terakhir agar trailing
# gap juga bisa diisi oleh Forward Fill.

def create_daily_series(series):
    start_date = series.index.min()
    end_date = series.index.max() + pd.Timedelta(days=7)

    date_range = pd.date_range(
        start=start_date,
        end=end_date,
        freq='D'
    )

    return series.reindex(date_range)


daily_series = (
    daily_observed
    .set_index('tanggal')
    .groupby(
        group_cols,
        observed=True
    )['harga_observasi']
    .apply(create_daily_series)
)

daily_series.index = daily_series.index.set_names(
    group_cols + ['tanggal']
)

prices_daily_ffill = (
    daily_series
    .rename('harga_observasi')
    .reset_index()
)



# ==========================================================
# 5. FORWARD FILL MAKSIMAL 7 HARI
# ==========================================================

prices_daily_ffill['harga_ffill'] = (
    prices_daily_ffill
    .groupby(
        group_cols,
        observed=True
    )['harga_observasi']
    .ffill(limit=7)
)

# Tandai harga yang berasal dari hasil imputasi
prices_daily_ffill['is_imputed'] = (
    prices_daily_ffill['harga_observasi'].isna()
    & prices_daily_ffill['harga_ffill'].notna()
)


# ==========================================================
# 6. RINGKASAN HASIL FORWARD FILL
# ==========================================================

total_daily_rows = len(prices_daily_ffill)

original_days = int(
    prices_daily_ffill['harga_observasi'].notna().sum()
)

imputed_days = int(
    prices_daily_ffill['is_imputed'].sum()
)

remaining_daily_missing = int(
    prices_daily_ffill['harga_ffill'].isna().sum()
)

print("\nRINGKASAN FORWARD FILL")
print("-" * 40)
print("Total baris kalender:", f"{total_daily_rows:,}")
print("Hari dengan harga asli:", f"{original_days:,}")
print("Hari berhasil diimputasi:", f"{imputed_days:,}")
print("Hari yang masih kosong:", f"{remaining_daily_missing:,}")


# Ringkasan per komoditas
ffill_summary = (
    prices_daily_ffill
    .groupby(
        'komoditas_normalisasi',
        observed=True
    )
    .agg(
        total_hari=('tanggal', 'size'),
        hari_asli=('harga_observasi', 'count'),
        hari_imputasi=('is_imputed', 'sum'),
        hari_masih_kosong=('harga_ffill', lambda x: x.isna().sum())
    )
    .sort_values('hari_imputasi', ascending=False)
)

display(ffill_summary)


# ==========================================================
# 7. BENTUK MATRIKS SEBELUM DAN SESUDAH FFILL
# ==========================================================

regions = sorted(
    clustering_source_ffill['kode_daerah'].dropna().unique()
)

commodities = sorted(
    clustering_source_ffill[
        'komoditas_normalisasi'
    ].dropna().unique()
)

# Matriks median berdasarkan observasi asli
clustering_data_before_ffill = (
    clustering_source_ffill
    .pivot_table(
        index='kode_daerah',
        columns='komoditas_normalisasi',
        values='harga',
        aggfunc='median',
        observed=True
    )
    .reindex(
        index=regions,
        columns=commodities
    )
)

# Matriks median berdasarkan harga harian setelah ffill
clustering_data_ffill = (
    prices_daily_ffill
    .dropna(subset=['harga_ffill'])
    .groupby(
        group_cols,
        observed=True
    )['harga_ffill']
    .median()
    .unstack('komoditas_normalisasi')
    .reindex(
        index=regions,
        columns=commodities
    )
)

# Pastikan nama index dan kolom konsisten
clustering_data_ffill.index.name = 'kode_daerah'
clustering_data_ffill.columns.name = 'komoditas_normalisasi'


# ==========================================================
# 8. EVALUASI PERUBAHAN MATRIKS
# ==========================================================

missing_before = int(
    clustering_data_before_ffill.isna().sum().sum()
)

missing_after = int(
    clustering_data_ffill.isna().sum().sum()
)

valid_cells = (
    clustering_data_before_ffill.notna()
    & clustering_data_ffill.notna()
)

absolute_difference = (
    clustering_data_ffill
    - clustering_data_before_ffill
).abs()

changed_cells = int(
    (
        (absolute_difference > 1e-9)
        & valid_cells
    ).sum().sum()
)

print("\nPERBANDINGAN MATRIKS CLUSTERING")
print("-" * 40)
print("Ukuran matriks:", clustering_data_ffill.shape)
print("Missing sebelum ffill:", missing_before)
print("Missing setelah ffill:", missing_after)
print("Nilai median yang berubah:", changed_cells)


# ==========================================================
# 9. ANALISIS MISSING VALUE SETELAH FFILL
# ==========================================================

missing_by_commodity_ffill = pd.DataFrame({
    'jumlah_missing': clustering_data_ffill.isna().sum(),
    'persentase_missing': (
        clustering_data_ffill.isna().mean() * 100
    )
}).sort_values(
    'jumlah_missing',
    ascending=False
)

missing_per_region_ffill = (
    clustering_data_ffill.isna().sum(axis=1)
)

print("\nWilayah tanpa missing:",
      int((missing_per_region_ffill == 0).sum()))

print("Wilayah dengan maksimal 2 missing:",
      int((missing_per_region_ffill <= 2).sum()))

print("Wilayah dengan lebih dari 3 missing:",
      int((missing_per_region_ffill > 3).sum()))

display(missing_by_commodity_ffill.round(2))


# Daftar pasangan wilayah-komoditas yang tetap missing
missing_pairs_after_ffill = (
    clustering_data_ffill.isna()
    .stack()
    .loc[lambda x: x]
    .rename('missing')
    .reset_index()
    .drop(columns='missing')
)


# ==========================================================
# 10. SIMPAN OUTPUT
# ==========================================================

clustering_data_ffill.to_csv(
    OUTPUT_DIR / 'clustering_matrix_after_ffill.csv',
    encoding='utf-8-sig'
)

ffill_summary.to_csv(
    OUTPUT_DIR / 'ffill_summary_by_commodity.csv',
    encoding='utf-8-sig'
)

missing_by_commodity_ffill.to_csv(
    OUTPUT_DIR / 'missing_value_after_ffill.csv',
    encoding='utf-8-sig'
)

missing_pairs_after_ffill.to_csv(
    OUTPUT_DIR / 'clustering_missing_pairs_after_ffill.csv',
    index=False,
    encoding='utf-8-sig'
)

print("\nSeluruh output Forward Fill berhasil disimpan.")


Jumlah komoditas: 32
Jumlah baris sumber: 4,595,223
Jumlah observasi harian: 4,595,223

RINGKASAN FORWARD FILL
----------------------------------------
Total baris kalender: 7,125,898
Hari dengan harga asli: 4,595,223
Hari berhasil diimputasi: 2,467,456
Hari yang masih kosong: 63,219


,total_hari,hari_asli,hari_imputasi,hari_masih_kosong
komoditas_normalisasi,,,,
TELUR AYAM RAS,352880,229212,121174,2494
BAWANG MERAH,352200,228810,120928,2462
DAGING AYAM RAS,350579,227699,120388,2492
TEPUNG TERIGU,348104,226277,119637,2190
BAWANG PUTIH,342429,222398,117720,2311
GULA PASIR,342289,222055,117409,2825
BERAS PREMIUM,330215,214207,113374,2634
BERAS MEDIUM,322847,209182,110833,2832
CABAI RAWIT,316441,205056,108599,2786



PERBANDINGAN MATRIKS CLUSTERING
----------------------------------------
Ukuran matriks: (544, 32)
Missing sebelum ffill: 2270
Missing setelah ffill: 2270
Nilai median yang berubah: 1018

Wilayah tanpa missing: 33
Wilayah dengan maksimal 2 missing: 253
Wilayah dengan lebih dari 3 missing: 207


,jumlah_missing,persentase_missing
komoditas_normalisasi,,
IKAN KEMBUNG/IKAN GEMBUNG/ IKAN BANYAR/IKAN GEMBOLO/ IKAN ASO-ASO,430,79.04
IKAN KEMBUNG,417,76.65
SUSU BUBUK UNTUK BALITA,186,34.19
SUSU BUBUK,161,29.60
UDANG BASAH,99,18.20
JERUK,87,15.99
PISANG,73,13.42
TEMPE,59,10.85
DAGING SAPI,57,10.48



Seluruh output Forward Fill berhasil disimpan.
